# Modelado y XAI — Alta contaminación por PM2.5 (Rol B)

**Proyecto:** *Dos Limas, un mismo cielo* — Contaminación del aire en Lima Metropolitana.

Este notebook documenta el **Panel 2 (predictivo)**. Reutiliza el pipeline de Rol A
(`preprocessing.cargar_y_limpiar`) y el módulo `models.py`; no reimplementa nada, para
que el notebook y el dashboard produzcan **exactamente los mismos resultados**.

**Decisiones (justificadas en `models.py`):**

- Etiqueta `y = (pm_25 > ECA_PM25)`, con `ECA_PM25 = 50` µg/m³ como constante única.
- Features = los 5 contaminantes restantes (se excluye `pm_25` para evitar fuga).
- Solo etiqueta **real** (`pm_25_imputado == False`).
- Sin escalado (RF/XGBoost son invariantes a escala).
- Desbalance ~92/8 → se compara `class_weight`, `scale_pos_weight` y SMOTE.
- `SEED = 96` importado de Rol A (reproducibilidad de todo el equipo).

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")

import pandas as pd
import models as M
from preprocessing import cargar_y_limpiar

print("SEED:", M.SEED, "| ECA_PM25:", M.ECA_PM25, "| FEATURES:", M.FEATURES)

## 1. Carga del dataset limpio (Rol A)

In [ ]:
df = cargar_y_limpiar(str(M.RUTA_DATOS))
print("shape:", df.shape)
df[["estacion", "fecha_hora", *M.CONTAMINANTES, "pm_25_imputado"]].head()

## 2. Construcción de la etiqueta binaria

`construir_dataset_modelado` filtra a PM2.5 medido, deriva `y` y arma `X` con las 5
features. Las aserciones internas garantizan que `pm_25` no se cuele como feature.

In [ ]:
X, y = M.construir_dataset_modelado(df)
print("X:", X.shape, "| features:", list(X.columns))
print(f"positivos (alta): {y.mean()*100:.1f}%  ->  desbalance que motiva SMOTE/class_weight")
y.value_counts()

## 3. Partición train/test (estratificada, 70/30)

In [ ]:
X_train, X_test, y_train, y_test = M.dividir(X, y)
print("train:", len(X_train), "| test:", len(X_test))
print("balance en test:", round(y_test.mean()*100, 2), "% positivos")

## 4. Entrenamiento: RF vs XGBoost vs RF+SMOTE

Tres estrategias frente al desbalance: `class_weight='balanced'` (RF),
`scale_pos_weight` (XGBoost, mecanismo nativo) y **SMOTE** (sobremuestreo sintético,
solo en train). *Nota: entrenar los tres puede tardar algunos minutos.*

In [ ]:
salida = M.entrenar_y_evaluar_todo(df)
salida["tabla"]

## 5. Comparación y elección del mejor modelo

Se ordena por **F1 de la clase 'alta'** (la minoritaria y la relevante). El costo de un
**falso negativo** (no avisar una hora contaminada) es mayor que el de un falso positivo,
así que priorizamos recall/F1 de esa clase sobre el accuracy global.

In [ ]:
mejor = salida["mejor"]
print("Mejor modelo:", mejor)
for clave, r in salida["resultados"].items():
    print(f"{clave:16s} F1_alta={r['clase_1_alta']['f1']:.3f} "
          f"recall_alta={r['clase_1_alta']['recall']:.3f} roc_auc={r['roc_auc']:.3f}")

## 6. Matriz de confusión

TP/TN/FP/FN del mejor modelo. La lectura clave: ¿cuántas horas realmente contaminadas
se nos escapan (FN) vs. cuántas falsas alarmas (FP)?

In [ ]:
from pathlib import Path
from IPython.display import Image
r_mejor = salida["resultados"][mejor]
print(r_mejor["matriz_confusion"])
M.figura_matriz_confusion(r_mejor, Path("../models/confusion_matrix.png"),
                          titulo=f"Matriz de confusión — {mejor}")
Image("../models/confusion_matrix.png")

## 7. Umbral de decisión (modificación en vivo)

Bajar el umbral de 0.50 a 0.30 reetiqueta **sin reentrenar**: sube el recall de 'alta'
a costa de precisión. Es la respuesta a la pregunta típica *'recall bajo, ¿qué harías?'*.

In [ ]:
modelo = salida["modelos"][mejor]
for u in (0.50, 0.30):
    r = M.evaluar(modelo, salida["X_test"], salida["y_test"], umbral=u)
    print(f"umbral={u:.2f} -> recall_alta={r['clase_1_alta']['recall']:.3f} "
          f"precision_alta={r['clase_1_alta']['precision']:.3f} "
          f"f1_alta={r['clase_1_alta']['f1']:.3f}")

## 8. Interpretabilidad (SHAP)

`summary_plot` (global): qué contaminantes empujan la predicción hacia 'alta'.
`force_plot` (local): descomposición de una instancia concreta. Con esto respondemos
*'¿por qué el modelo predijo X para este caso?'* y discutimos correlación vs. causalidad
(p. ej. si `pm_10` domina, es esperable porque comparte fuentes de combustión con PM2.5).

In [ ]:
rutas = M.explicar_shap(salida["modelos"][mejor], salida["X_test"],
                        Path("../models"), prefijo=mejor)
Image(rutas["summary"])

In [ ]:
Image(rutas["force"])

## 9. Persistencia de artefactos

Se guardan modelos y métricas para el dashboard y el Reporte PDF. `metrics.json` es la
**fuente de verdad** de las cifras que se muestran en el Panel 2.

In [ ]:
import json
M.DIR_MODELOS.mkdir(exist_ok=True)
M.guardar_modelo(salida["modelos"]["rf_classweight"], M.DIR_MODELOS/"rf.pkl")
M.guardar_modelo(salida["modelos"]["xgb_scaleposw"], M.DIR_MODELOS/"xgb.pkl")
M.guardar_modelo(salida["modelos"]["rf_smote"], M.DIR_MODELOS/"rf_smote.pkl")
metrics = M._construir_metrics_json(salida)
(M.DIR_MODELOS/"metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Artefactos escritos en", M.DIR_MODELOS)
metrics

---
**Reproducir todo de una sola vez** (equivalente a este notebook):

```bash
uv run python src/models.py
```